In [2]:
import duckdb
con = duckdb.connect()

con.sql("INSTALL tpch")         
con.sql("LOAD tpch")        
con.sql("CALL dbgen(sf=0.01)")    

con.sql("SHOW TABLES").df()     


,name
0,customer
1,lineitem
2,nation
3,orders
4,part
5,partsupp
6,region
7,supplier


In [3]:
con.sql("SELECT * FROM orders LIMIT 5")

┌────────────┬───────────┬───────────────┬───────────────┬─────────────┬─────────────────┬─────────────────┬────────────────┬───────────────────────────────────────────────────────────────────────────┐
│ o_orderkey │ o_custkey │ o_orderstatus │ o_totalprice  │ o_orderdate │ o_orderpriority │     o_clerk     │ o_shippriority │                                 o_comment                                 │
│   int64    │   int64   │    varchar    │ decimal(15,2) │    date     │     varchar     │     varchar     │     int32      │                                  varchar                                  │
├────────────┼───────────┼───────────────┼───────────────┼─────────────┼─────────────────┼─────────────────┼────────────────┼───────────────────────────────────────────────────────────────────────────┤
│          1 │       370 │ O             │     172799.49 │ 1996-01-02  │ 5-LOW           │ Clerk#000000951 │              0 │ ly express platelets. deposits acc                                

# Warm up queries

### Q1: Rank each customer's orders by price
Window function: `DENSE_RANK` per customer, highest `o_totalprice` first (customers 1 to 10).

In [4]:
con.sql("""
    SELECT *,
        DENSE_RANK() OVER(PARTITION BY o_custkey ORDER BY o_totalprice DESC) as price_rank
    FROM orders
    WHERE o_custkey <= 10
    ORDER BY o_custkey, price_rank
""")

┌────────────┬───────────┬───────────────┬───────────────┬─────────────┬─────────────────┬─────────────────┬────────────────┬───────────────────────────────────────────────────────────────────────────────┬────────────┐
│ o_orderkey │ o_custkey │ o_orderstatus │ o_totalprice  │ o_orderdate │ o_orderpriority │     o_clerk     │ o_shippriority │                                   o_comment                                   │ price_rank │
│   int64    │   int64   │    varchar    │ decimal(15,2) │    date     │     varchar     │     varchar     │     int32      │                                    varchar                                    │   int64    │
├────────────┼───────────┼───────────────┼───────────────┼─────────────┼─────────────────┼─────────────────┼────────────────┼───────────────────────────────────────────────────────────────────────────────┼────────────┤
│       9154 │         1 │ O             │     357345.46 │ 1997-06-23  │ 4-NOT SPECIFIED │ Clerk#000000328 │              0 

### Q2: Running total of fulfilled orders per customer
Cumulative `SUM` of `o_totalprice` over time, only for fulfilled orders (`o_orderstatus = 'F'`). The filter runs before the window.

In [5]:
con.sql("""
    SELECT o_custkey, o_orderkey, o_orderdate, o_totalprice,
           SUM(o_totalprice) OVER(PARTITION BY o_custkey ORDER BY o_orderdate, o_orderkey) AS running_sum
    FROM orders
    WHERE o_custkey <= 10 AND o_orderstatus = 'F'
    ORDER BY o_custkey, o_orderdate, o_orderkey
""")

┌───────────┬────────────┬─────────────┬───────────────┬───────────────┐
│ o_custkey │ o_orderkey │ o_orderdate │ o_totalprice  │  running_sum  │
│   int64   │   int64    │    date     │ decimal(15,2) │ decimal(38,2) │
├───────────┼────────────┼─────────────┼───────────────┼───────────────┤
│         1 │      31653 │ 1993-06-05  │     152411.41 │     152411.41 │
│         1 │      43879 │ 1993-08-13  │      83095.85 │     235507.26 │
│         1 │      52263 │ 1994-05-08  │      51134.82 │     286642.08 │
│         2 │       6980 │ 1993-02-19  │     170842.93 │     170842.93 │
│         2 │      29408 │ 1993-05-03  │     154867.09 │     325710.02 │
│         2 │      10563 │ 1993-09-30  │     143707.70 │     469417.72 │
│         2 │      29956 │ 1994-08-15  │     116247.57 │     585665.29 │
│         2 │      44962 │ 1994-12-29  │      45657.87 │     631323.16 │
│         4 │      22466 │ 1992-03-29  │     130160.51 │     130160.51 │
│         4 │      10688 │ 1992-05-13  │      43453

### Q3: Change from the customer's previous order
`LAG` returns the previous order's price within each customer. The first order has no previous one, so it is `NULL`. A CTE keeps the difference calculation readable.

In [6]:
con.sql("""
    WITH with_prev AS (
        SELECT o_custkey, o_orderkey, o_orderdate, o_totalprice,
               LAG(o_totalprice) OVER(PARTITION BY o_custkey ORDER BY o_orderdate, o_orderkey) AS prev_amount
        FROM orders
        WHERE o_custkey <= 10
    )
    SELECT *, o_totalprice - prev_amount AS diff
    FROM with_prev
    ORDER BY o_custkey, o_orderdate, o_orderkey
""")

┌───────────┬────────────┬─────────────┬───────────────┬───────────────┬───────────────┐
│ o_custkey │ o_orderkey │ o_orderdate │ o_totalprice  │  prev_amount  │     diff      │
│   int64   │   int64    │    date     │ decimal(15,2) │ decimal(15,2) │ decimal(16,2) │
├───────────┼────────────┼─────────────┼───────────────┼───────────────┼───────────────┤
│         1 │      31653 │ 1993-06-05  │     152411.41 │          NULL │          NULL │
│         1 │      43879 │ 1993-08-13  │      83095.85 │     152411.41 │     -69315.56 │
│         1 │      52263 │ 1994-05-08  │      51134.82 │      83095.85 │     -31961.03 │
│         1 │      53283 │ 1995-10-29  │     165928.33 │      51134.82 │     114793.51 │
│         1 │      24322 │ 1997-01-29  │     231040.44 │     165928.33 │      65112.11 │
│         1 │      36422 │ 1997-03-04  │     270087.44 │     231040.44 │      39047.00 │
│         1 │       9154 │ 1997-06-23  │     357345.46 │     270087.44 │      87258.02 │
│         1 │      14

In [7]:
con.sql("""
    SELECT *
    FROM customer
    limit 5
""")

┌───────────┬────────────────────┬────────────────────────────────┬─────────────┬─────────────────┬───────────────┬──────────────┬────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│ c_custkey │       c_name       │           c_address            │ c_nationkey │     c_phone     │   c_acctbal   │ c_mktsegment │                                               c_comment                                                │
│   int64   │      varchar       │            varchar             │    int32    │     varchar     │ decimal(15,2) │   varchar    │                                                varchar                                                 │
├───────────┼────────────────────┼────────────────────────────────┼─────────────┼─────────────────┼───────────────┼──────────────┼────────────────────────────────────────────────────────────────────────────────────────────────────────┤
│         1 │ Customer#000000001 │ j5JsirBM9PsCy0O1m    

### Q4: Customers who never placed an order (anti-join)
`LEFT JOIN` customers to orders, then keep rows where the order key is `NULL` (no match found). Customers 1 to 10 only. 

In [8]:
con.sql("""
    SELECT c.c_custkey, c.c_name
    FROM customer c
    LEFT JOIN orders o on c.c_custkey = o.o_custkey
    WHERE c.c_custkey<=10 AND o.o_orderkey is NULL
    ORDER BY c.c_custkey
""")

┌───────────┬────────────────────┐
│ c_custkey │       c_name       │
│   int64   │      varchar       │
├───────────┼────────────────────┤
│         3 │ Customer#000000003 │
│         6 │ Customer#000000006 │
│         9 │ Customer#000000009 │
└───────────┴────────────────────┘

### Q5: Row count after joining orders to lineitem
One order has many order lines, so joining multiplies rows.

In [16]:
con.sql("""
SELECT
    (SELECT COUNT(*) FROM orders) AS orders_rows,
    (SELECT COUNT(*)
     FROM orders o
     JOIN lineitem l ON o.o_orderkey = l.l_orderkey) AS joined_rows
""")

┌─────────────┬─────────────┐
│ orders_rows │ joined_rows │
│    int64    │    int64    │
├─────────────┼─────────────┤
│       15000 │       60175 │
└─────────────┴─────────────┘

### Q6: Join explosion inflates a SUM
`o_totalprice` lives on the orders side. After the join it is repeated once per order line, so summing it counts each order several times.

In [19]:
con.sql("""
    SELECT 
        (SELECT SUM(o_totalprice) FROM orders) AS order_rows,
        (SELECT SUM(o.o_totalprice)
        FROM orders o
        JOIN lineitem l ON o.o_orderkey = l.l_orderkey) AS inflated_total
""")

┌───────────────┬────────────────┐
│  order_rows   │ inflated_total │
│ decimal(38,2) │ decimal(38,2)  │
├───────────────┼────────────────┤
│ 2127396830.02 │ 10645296330.84 │
└───────────────┴────────────────┘